# Phase 0 · Configure the V3 pipeline

Set the organism groups and every acquisition, decision, logging, and embedding parameter used by Phases 1–5. This follows the V2 configuration notebook: settings are visible in phase-specific cells and saved to shared YAML files. Phase notebooks read these files when they run.

Phase 1 always builds the complete master taxonomy. `organism_groups` selects which groups Phases 2–5 process. Select one or more of `bacteria`, `fungi`, and `viruses`.


In [ ]:
from pathlib import Path
import os
import sys
import yaml
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml

DEFAULT_CONFIG_PATH = PROJECT_ROOT / "configs/default.yaml"
TAXONOMY_CONFIG_PATH = PROJECT_ROOT / "configs/taxonomy/source_files.yaml"
CONFIG = load_yaml(DEFAULT_CONFIG_PATH)
TAXONOMY_CONFIG = load_yaml(TAXONOMY_CONFIG_PATH)
print("Project root:", PROJECT_ROOT)
print("Shared run settings:", DEFAULT_CONFIG_PATH)
print("Taxonomy source settings:", TAXONOMY_CONFIG_PATH)


## Project and organism-group selection

In [ ]:
# Phase 1 builds the full taxonomy; this selection controls Phases 2–5.
PROJECT_SETTINGS = {
    "id": "pathogen-atlas-v3",
    "schema_version": 1,
    "domains": ["bacteria", "fungi", "viruses"],
}
RUN_SETTINGS = {
    "organism_groups": ["viruses"],  # choose any non-empty subset of the supported groups
}


## Phase 1 · taxonomy inputs

In [ ]:
# Paths are relative to the V3 project root. Releases and source identifiers are retained in output provenance.
TAXONOMY_SOURCES = {
    "bacteria": {
        "adapter": "lpsn",
        "path": "data/migration/version2_snapshot/01_species/release/species_canonical.parquet",
        "release": "migrated-v2-lpsn-local-capture",
    },
    "fungi": {
        "adapter": "species_fungorum",
        "path": "data/source_taxonomy/fungi/SFPK_Published_Names_Export_16102026.zip",
        "release": "Species Fungorum snapshot archived 12 October 2026; export 16 October 2026",
        "source_url": "https://sftp.kew.org/pub/data-repositories/SpeciesFungorum/SFPK%20-%20Published%20Names%20Export%2016102026.zip",
        "license": "CC-BY-3.0",
    },
    "viruses": {
        "adapter": "ictv_msl",
        "path": "data/source_taxonomy/viruses/ICTV_Master_Species_List_2025_MSL41.v1.xlsx",
        "release": "ICTV MSL41 (2025 taxonomy; ratified February 2026), v1",
        "source_url": "https://ictv.global/sites/default/files/MSL/ICTV_Master_Species_List_2025_MSL41.v1.xlsx",
        "license": "CC-BY-4.0",
        "variant_metadata_adapter": "ictv_vmr",
        "variant_metadata_path": "data/source_taxonomy/viruses/VMR_MSL41.v1.20260729.xlsx",
        "variant_metadata_release": "ICTV VMR MSL41 v1, 2026-07-29",
        "variant_metadata_url": "https://ictv.global/sites/default/files/VMR/VMR_MSL41.v1.20260729.xlsx",
    },
}


## Shared paths

In [ ]:
PATH_SETTINGS = {
    "data_root": "data",
    "migration_source": "data/migration/version2_snapshot",
    "taxonomy_release": "data/01_taxonomy/release",
    "pubmed_cache": "data/02_pubmed/cache",
    "pubmed_release": "data/02_pubmed/release",
    "pubmed_domain_root": "data/02_pubmed/domains",
    "decision_ledger": "data/03_decisions/parsed/decision_ledger.parquet",
    "raw_responses": "data/03_decisions/raw_responses/jev.jsonl",
    "release_root": "data/04_release",
    "embedding_root": "data/05_embeddings",
}


## Phase 2 · PubMed acquisition

In [ ]:
PUBMED_SETTINGS = {
    "email_env": "NCBI_EMAIL",
    "api_key_env": "NCBI_API_KEY",
    "tool": "pathogen_atlas_v3",
    "max_workers": 10,
    "request_delay_seconds": None,  # None selects API-key-aware pacing
    "batch_size": 200,
    "retmax": 10000,
    "checkpoint_every": 100,
    "max_calls": None,              # None means all pending taxon searches
    "max_fetch_batches": None,      # None means fetch all pending abstract batches
}


## Phase 3 · Jev decisions and feature scoring

In [ ]:
DECISION_SETTINGS = {
    "provider": "openrouter_jev",
    "model": "typesafe/jev-1.14",
    "endpoint": "https://openrouter.ai/api/alpha/decisions",
    "timeout_seconds": 90,
    "retries": 3,
    "backoff_seconds": 1.0,
    "max_workers": 32,
    "max_calls": None,  # cap new work for a pilot; None processes every pending decision
    "pathogen_question_version": "human-disease-attribution-v3.1",
    "feature_question_set_version": "multi-domain-features-v3.1",
    "pathogen_threshold": 0.75,
    "feature_yes_threshold": 0.75,
    "release_evaluation_approved": False,
    "cost_estimate": {
        "expected_pathogen_fraction": 0.5,
        "input_tokens_per_gate_call": 1200,
        "input_tokens_per_feature_call": 4200,
        "output_tokens_per_call": 500,
        "input_usd_per_million_tokens": 0.042,
        "output_usd_per_million_tokens": 0.0,
    },
}


## Phase 5 · IMSTE embedding

In [ ]:
EMBEDDING_SETTINGS = {
    "cross_domain_feature_profile": "shared-core-v1",
    "within_domain_feature_profile": "domain-specific-v1",
    "zero_impute_not_applicable": False,
    "seed": 42,
    "n_msts": 15,
    "n_epochs": 300,
    "mst_method": "famst",
    "mst_neighbors": 15,
    "mst_inter_component_edges": 5,
    "mst_max_neighbors": None,
    "negative_ratio": 5,
    "lambda_rep": 0.5,
    "learning_rate": 0.05,
    "batch_size": 32768,
    "epsilon": 0.0001,
    "logistic_margin": 1.0,
    "logistic_temperature": 0.5,
    "mst_representatives_per_component": 10,
    "device": "auto",
}


## Logs, checkpoints, and ETA

In [ ]:
LOGGING_SETTINGS = {
    "progress_every": 100,  # progress message cadence for network/decision work
    "show_eta": True,
    "show_openrouter_balance": True,
}


## Review and save shared configuration

In [ ]:
# Assemble the settings exactly as consumed by the phase modules.
CONFIG.update({
    "project": PROJECT_SETTINGS,
    "run": RUN_SETTINGS,
    "paths": PATH_SETTINGS,
    "pubmed": PUBMED_SETTINGS,
    "decisions": DECISION_SETTINGS,
    "embedding": EMBEDDING_SETTINGS,
    "logging": LOGGING_SETTINGS,
})
TAXONOMY_CONFIG["sources"] = TAXONOMY_SOURCES

SELECTED_GROUPS = selected_organism_groups(CONFIG)
print("Selected groups for Phases 2–5:", ", ".join(SELECTED_GROUPS))
print("Phase 2 query cap:", PUBMED_SETTINGS["max_calls"], "| abstract batch cap:", PUBMED_SETTINGS["max_fetch_batches"])
print("Phase 3 call cap:", DECISION_SETTINGS["max_calls"], "| concurrency:", DECISION_SETTINGS["max_workers"])
print("Progress interval:", LOGGING_SETTINGS["progress_every"], "| ETA display:", LOGGING_SETTINGS["show_eta"])
display(pd.DataFrame([
    {"phase": "1 · Taxonomy", "scope": "complete master taxonomy", "config": str(TAXONOMY_CONFIG_PATH)},
    {"phase": "2 · PubMed", "scope": ", ".join(SELECTED_GROUPS), "config": str(DEFAULT_CONFIG_PATH)},
    {"phase": "3 · Jev decisions", "scope": ", ".join(SELECTED_GROUPS), "config": str(DEFAULT_CONFIG_PATH)},
    {"phase": "4 · Release tables", "scope": "saved decisions", "config": str(DEFAULT_CONFIG_PATH)},
    {"phase": "5 · IMSTE", "scope": ", ".join(SELECTED_GROUPS), "config": str(DEFAULT_CONFIG_PATH)},
]))

SAVE_CONFIGURATION = True
if SAVE_CONFIGURATION:
    DEFAULT_CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True), encoding="utf-8")
    TAXONOMY_CONFIG_PATH.write_text(yaml.safe_dump(TAXONOMY_CONFIG, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print("Saved run settings to", DEFAULT_CONFIG_PATH)
    print("Saved taxonomy inputs to", TAXONOMY_CONFIG_PATH)
else:
    print("Review only; set SAVE_CONFIGURATION=True to write these settings.")
